# Practice: Building and Training a CNN with PyTorch

Use this practice after Section 6.

- Load the digit images and inspect the image tensors.
- Predict batch shapes and verify what happens in the final batch.
- Trace a batch through convolution, pooling and classification.
- Train a CNN and inspect errors on held-out images.

Run the notebook from top to bottom in a fresh runtime. It downloads the digit images if needed. A GPU is optional.

Use Colab's standard Python runtime. For local Jupyter, install missing packages with `python -m pip install torch torchvision scikit-learn`.

[Open in Colab](https://colab.research.google.com/github/wlos/ai_with_python_course/blob/main/16-Deep-Learning-and-Image-Processing/practice/Practice%20CNN%20Training%20with%20PyTorch.ipynb)


## 1. Inspect image transforms

The PNGs are grouped in folders named `0` to `9`. Before running the cell, predict which transform sets the number of channels and which changes the pixel value range. Check the class mapping and explain why the first label is zero.


In [ ]:
from os import makedirs, replace
from os.path import isfile
from shutil import unpack_archive
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

DIGIT_ARCHIVE = "sample_data/digit_images.zip"
DIGIT_URL = (
    "https://raw.githubusercontent.com/wlos/AI_SS/main/"
    "13-Computer-Vision/sample_data/digit_images.zip"
)

if not isfile(DIGIT_ARCHIVE):
    makedirs("sample_data", exist_ok=True)
    urlretrieve(DIGIT_URL, DIGIT_ARCHIVE + ".part")
    replace(DIGIT_ARCHIVE + ".part", DIGIT_ARCHIVE)  # Keep only a completed download.

# Keep the extracted images available while ImageFolder reads them on demand.
digit_files = TemporaryDirectory()
unpack_archive(DIGIT_ARCHIVE, digit_files.name)
digit_root = f"{digit_files.name}/sample_data/digit_images"

from torchvision import datasets, transforms

DIGIT_SIDE = 28

digit_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Keep a single intensity channel.
    transforms.Resize((DIGIT_SIDE, DIGIT_SIDE)),
    transforms.ToTensor(),  # Convert pixels to float values in [0, 1].
])
digit_dataset = datasets.ImageFolder(digit_root, transform=digit_transform)
sample_image, sample_label = digit_dataset[0]
print(f"Classes: {digit_dataset.class_to_idx}")
print(f"Image shape: {tuple(sample_image.shape)} | Label: {sample_label}")
assert len(digit_dataset) == 240 and len(digit_dataset.classes) == 10
assert tuple(sample_image.shape) == (1, DIGIT_SIDE, DIGIT_SIDE)


## 2. Prepare image batches

The stratified split gives 192 training images and 48 test images. Set `DIGIT_BATCH_SIZE` to **20** in the supplied code. Predict the first and last batch sizes for each split. Check your predictions with the printed results and explain why the test loader keeps the image order fixed.


In [ ]:
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

DIGIT_SEED = 42
DIGIT_TEST_FRACTION = 0.2
DIGIT_BATCH_SIZE = 16

# Split image indices so the dataset still opens and transforms each file on demand.
digit_train_indices, digit_test_indices = train_test_split(
    list(range(len(digit_dataset))),
    test_size=DIGIT_TEST_FRACTION,
    random_state=DIGIT_SEED,
    stratify=digit_dataset.targets,
)
digit_train = Subset(digit_dataset, digit_train_indices)
digit_test = Subset(digit_dataset, digit_test_indices)
digit_train_loader = DataLoader(
    digit_train, batch_size=DIGIT_BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(DIGIT_SEED),
)
digit_test_loader = DataLoader(digit_test, batch_size=DIGIT_BATCH_SIZE, shuffle=False)

batch_images, batch_labels = next(iter(digit_test_loader))
print(f"Training images: {len(digit_train)} | Test images: {len(digit_test)}")
print(f"Image batch: {tuple(batch_images.shape)} | Label batch: {tuple(batch_labels.shape)}")
print(f"Batch labels: {batch_labels.tolist()}")

# Inspect every batch size, including the final batch.
print("Training batch sizes:", [len(labels) for _, labels in digit_train_loader])
print("Test batch sizes:", [len(labels) for _, labels in digit_test_loader])


## 3. Check the CNN feature shapes

Before running the cell, write the expected channel, height and width after `Conv2d` and after `MaxPool2d`. Explain which dimensions change when you double the number of convolution filters. The classifier flattens the feature maps for each image and returns one logit per digit.


In [ ]:
from torch import nn


class DigitCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()  # Keep the batch dimension separate.
        pooled_side = DIGIT_SIDE // 2
        self.head = nn.Linear(8 * pooled_side * pooled_side, num_classes)

    def forward(self, images):
        features = self.relu(self.conv(images))
        pooled = self.pool(features)
        return self.head(self.flatten(pooled))


torch.manual_seed(DIGIT_SEED)  # Reproduce the model's initial weights.
digit_class_count = len(digit_dataset.classes)
digit_cnn = DigitCNN(digit_class_count)

# Pass one batch through the untrained model to check tensor shapes.
with torch.no_grad():
    feature_maps = digit_cnn.relu(digit_cnn.conv(batch_images))
    pooled_maps = digit_cnn.pool(feature_maps)
    digit_logits = digit_cnn(batch_images)
print(f"After convolution and ReLU: {tuple(feature_maps.shape)}")
print(f"After max pooling: {tuple(pooled_maps.shape)}")
print(f"Output logits: {tuple(digit_logits.shape)}")

# Verify that pooling halves the spatial dimensions and preserves channels.
assert pooled_maps.shape[1] == feature_maps.shape[1]
assert pooled_maps.shape[-2:] == (DIGIT_SIDE // 2, DIGIT_SIDE // 2)
assert digit_logits.shape == (len(batch_images), digit_class_count)


## 4. Train and inspect test predictions

For each training batch, identify the line that clears gradients, the line that calculates gradients and the line that updates the weights. Record the training loss after the first and last epochs.


In [ ]:
DIGIT_EPOCHS = 3
DIGIT_LEARNING_RATE = 0.001
digit_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
digit_loss = nn.CrossEntropyLoss()


def train_digit_epoch(model, loader, optimiser):
    '''Update trainable parameters and return mean loss per image.'''
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(digit_device), labels.to(digit_device)
        optimiser.zero_grad()  # Clear gradients left by the previous batch.
        loss = digit_loss(model(images), labels)
        loss.backward()  # Calculate gradients from the loss.
        optimiser.step()  # Apply an update to the optimiser's parameters.
        total_loss += loss.item() * len(labels)  # Weight each batch by its size.
    return total_loss / len(loader.dataset)


digit_cnn = digit_cnn.to(digit_device)
digit_optimiser = torch.optim.Adam(digit_cnn.parameters(), lr=DIGIT_LEARNING_RATE)
digit_cnn.train()
for epoch in range(DIGIT_EPOCHS):
    mean_loss = train_digit_epoch(digit_cnn, digit_train_loader, digit_optimiser)
    print(f"Simple CNN epoch {epoch + 1}: loss={mean_loss:.3f}")


Run the final cell and find one incorrectly classified image in the first test batch. Report its true and predicted digits. Explain why you would use a validation split from the training data to select a different epoch count or learning rate.


In [ ]:
def digit_accuracy(model, loader):
    '''Measure the fraction of test images classified correctly.'''
    model.eval()  # Use each layer's inference behaviour.
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            predicted = model(images.to(digit_device)).argmax(dim=1)
            correct += (predicted.cpu() == labels).sum().item()
    return correct / len(loader.dataset)


simple_digit_accuracy = digit_accuracy(digit_cnn, digit_test_loader)
print(f"Simple CNN test accuracy: {simple_digit_accuracy:.1%}")

# The model is in evaluation mode after calculating test accuracy.
with torch.no_grad():
    first_predictions = digit_cnn(batch_images.to(digit_device)).argmax(dim=1).cpu()
print("True digits:     ", batch_labels.tolist())
print("Predicted digits:", first_predictions.tolist())
